# M0 · 06 — Reductions & softmax

A **reduction** collapses an axis: `sum`, `mean`, `max`. The `dim=` argument
chooses *which* axis to collapse — the same idea that makes
`F.softmax(weights, dim=-1)` normalise over the right axis.

Getting `dim` right is one of the most common beginner pain points, so we drill it.

In [1]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

In [2]:
m = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
m  # shape (2, 3)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

## 1. Sum everything

`m.sum()` with no `dim` collapses **all** axes to a single number.
Compute the total sum of `m` (should be 21).

In [3]:
total = m.sum()
total

tensor(21.)

In [4]:
check('total sum', total, torch.tensor(21.0))

✅ total sum


True

## 2. `dim=0` collapses the rows (sum down each column)

`dim=0` means "remove axis 0" → you get one value per **column**.
`m.sum(dim=0)` → `[1+4, 2+5, 3+6] = [5, 7, 9]`.

Compute the column sums.

In [5]:
col_sums = m.sum(dim=0)
col_sums

tensor([5., 7., 9.])

In [6]:
check('column sums', col_sums, torch.tensor([5., 7., 9.]))

✅ column sums


True

## 3. `dim=1` (a.k.a. `dim=-1`) collapses the columns (sum across each row)

`dim=1` removes axis 1 → one value per **row**: `[1+2+3, 4+5+6] = [6, 15]`.
Since it's the last axis, `dim=-1` means the same thing here.

Compute the row sums using `dim=-1`.

In [7]:
row_sums = m.sum(dim=-1)
row_sums

tensor([ 6., 15.])

In [8]:
check('row sums', row_sums, torch.tensor([6., 15.]))

✅ row sums


True

## 4. `keepdim=True` — keep the collapsed axis as size 1

Normally reducing `(2,3)` over `dim=-1` gives `(2,)`. With `keepdim=True` you get
`(2,1)` — which then **broadcasts** cleanly back against the original. This is the
trick behind normalising: `m / m.sum(dim=-1, keepdim=True)`.

Compute row sums with `keepdim=True`; shape must be `(2, 1)`.

In [9]:
row_sums_kd = m.sum(dim=-1, keepdim=True)
row_sums_kd.shape

torch.Size([2, 1])

In [10]:
check_tensor('keepdim shape (2,1)', row_sums_kd, shape=(2, 1))

✅ keepdim shape (2,1)


True

## 5. Normalise each row to sum to 1 (a 'poor-man's softmax')

Divide `m` by its row sums (with `keepdim=True` so broadcasting works). Each row
of the result should sum to 1. This is exactly why `keepdim` matters.

In [16]:
norm = m / m.sum(dim=-1, keepdim=True)
# m is (2,3)
# m.sum is (2,1)
# tensor([[1., 2., 3.],
#         [4., 5., 6.]])
print(norm)
norm.sum(dim=-1)  # each ~1.0 #

tensor([[0.1667, 0.3333, 0.5000],
        [0.2667, 0.3333, 0.4000]])


tensor([1., 1.])

In [12]:
check('rows sum to 1', norm.sum(dim=-1), torch.tensor([1.0, 1.0]))

✅ rows sum to 1


True

## 6. The real softmax — `F.softmax(x, dim=-1)`

Softmax turns arbitrary scores into a probability distribution along one axis:
`exp(x) / sum(exp(x))`. The GPT calls `F.softmax(weights, dim=-1)` so each
query's attention over keys sums to 1.

Apply softmax to `scores` along the **last** axis; verify each row sums to 1.

In [18]:
from torch.nn import functional as F
scores = torch.tensor([[2.0, 1.0, 0.0],
                       [0.0, 0.0, 0.0]])
probs = F.softmax(scores, dim=-1)
probs

tensor([[0.6652, 0.2447, 0.0900],
        [0.3333, 0.3333, 0.3333]])

In [19]:
check('softmax rows sum to 1', probs.sum(dim=-1), torch.tensor([1.0, 1.0]))
check('uniform row is uniform', probs[1], torch.tensor([1/3, 1/3, 1/3]))

✅ softmax rows sum to 1
✅ uniform row is uniform


True

## ✅ Recap

- Reductions collapse an axis; `dim=` picks which one (`dim=0` → per column,
  `dim=-1` → per row).
- `keepdim=True` leaves a size-1 axis so the result broadcasts back.
- `x / x.sum(dim=-1, keepdim=True)` normalises rows; `softmax(dim=-1)` is the
  smooth probability version used for attention weights.

Next: **07 — masking & embedding lookup** (the causal mask and `nn.Embedding`).